In [ ]:
!pip install -q torch transformers huggingface_hub numpy pandas networkx matplotlib scikit-learn

## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer, set_seed,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
def all_seeds(seed=19):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

all_seeds(19)

### Load and Prepare LLM-Labeled Data

In [ ]:
df = pd.read_csv('/path/to/llm_labeled_balanced_final.csv')
df = df.rename(columns={'label': 'llm_label', 'confidence': 'llm_confidence'})

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['llm_label'].value_counts()}")

Total samples: 1179
Label distribution:
llm_label
positive    400
neutral     393
negative    386
Name: count, dtype: int64


## Clean Sentences (Remove Entity Markers)


In [ ]:
import re

def clean_sentence(text):
    """
    Standard cleanup for single-turn training data.
    Removes entity markers and speaker tags.
    """
    text = re.sub(r'\[Turn \d+\]', '', text)
    text = re.sub(r'\[SPEAKER\]', '', text)
    text = re.sub(r'\[/?E[12]\]', '', text)
    text = re.sub(r'Speaker \d+:', '', text)
    return ' '.join(text.split()).strip()

def extract_and_clean_target(text):
    """
    Special cleanup for the 150 manual samples.
    Extracts ONLY the Target Turn content before cleaning.
    """
    # Look for text inside (Target Turn: ...)
    match = re.search(r'\(Target Turn: (.*?)\)', text)
    if match:
        target_content = match.group(1)
        return clean_sentence(target_content)

    # If the format isn't found, just clean the whole text as fallback
    return clean_sentence(text)

## Map Labels to Integers


In [ ]:
label_map = {
    'positive': 0,
    'negative': 1,
    'neutral': 2
}

df['label'] = df['llm_label'].map(label_map)

print(f"\nLabel encoding:")
for label, idx in label_map.items():
    count = (df['label'] == idx).sum()
    print(f"  {label} ({idx}): {count} samples")


Label encoding:
  positive (0): 400 samples
  negative (1): 386 samples
  neutral (2): 393 samples


## Train/Val Split

In [ ]:
# Apply standard cleaning to your large training/balanced dataset
df['sentence_clean'] = df['sentence'].apply(clean_sentence)

train_df, val_df = train_test_split(
    df[['sentence_clean', 'label', 'llm_confidence']],
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)

print("Sample Cleaned Training Sentences:")
print(df['sentence_clean'].head(3))

print(f"\n✓ Train/Val split:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val:   {len(val_df)} samples")

# Show distribution
print(f"\n✓ Train label distribution:")
print(train_df['label'].value_counts())
print(f"\n✓ Val label distribution:")
print(val_df['label'].value_counts())

Sample Cleaned Training Sentences:
0    Oh my God, its happening. It's already started...
1    Yeah, Ross can't go so it's between my friend ...
2    Umm, I'm sorry Judy, I couldn't find that bowl...
Name: sentence_clean, dtype: object

✓ Train/Val split:
  Train: 943 samples
  Val:   236 samples

✓ Train label distribution:
label
0    320
2    314
1    309
Name: count, dtype: int64

✓ Val label distribution:
label
0    80
2    79
1    77
Name: count, dtype: int64


## Load RoBERTa-GoEmotions Model

In [ ]:
MODEL_NAME = "Lakssssshya/roberta-large-goemotions"

print(f"\nLoading {MODEL_NAME}...")

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,  # positive, negative, neutral
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True
)

print(f"✓ Model loaded: {model.num_parameters():,} parameters")


Loading Lakssssshya/roberta-large-goemotions...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/643 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Lakssssshya/roberta-large-goemotions and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded: 355,362,819 parameters


## Tokenize Data

In [ ]:
def tokenize_function(examples):
    """Tokenize sentences."""
    return tokenizer(
        examples['sentence_clean'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['sentence_clean', 'label']])
val_dataset = Dataset.from_pandas(val_df[['sentence_clean', 'label']])

# Tokenize
print("\nTokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✓ Tokenization complete")


Tokenizing datasets...


Map:   0%|          | 0/943 [00:00<?, ? examples/s]

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

✓ Tokenization complete


## Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Compute F1, accuracy, precision, recall."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_positive': f1_per_class[0],
        'f1_negative': f1_per_class[1],
        'f1_neutral': f1_per_class[2]
    }

## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir='/roberta-goemotions-1turn',
    report_to="none",

    # Training hyperparameters
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=8e-6,
    weight_decay=0.01,
    warmup_steps=100,

    # Evaluation
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=20,

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
)

print("\n✓ Training configuration set")


✓ Training configuration set


## Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.01
        )
    ]
)

print("✓ Trainer initialized")

✓ Trainer initialized


## Training

In [ ]:
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

train_result = trainer.train()

print("\n" + "="*70)
print("✓ TRAINING COMPLETE!")
print("="*70)
print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Total training time: {train_result.metrics['train_runtime']:.1f}s")


STARTING TRAINING


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Positive,F1 Negative,F1 Neutral
50,1.107700,1.096299,0.338983,0.176275,0.000000,0.025641,0.503185
100,1.002200,0.889873,0.635593,0.635829,0.591195,0.658228,0.658065
150,0.736800,0.668136,0.750000,0.749537,0.723684,0.765432,0.759494
200,0.546200,0.605686,0.758475,0.757749,0.717241,0.797386,0.758621
250,0.343200,0.748603,0.737288,0.737514,0.706667,0.761194,0.744681
300,0.279800,0.628519,0.834746,0.835626,0.847682,0.849673,0.809524
350,0.315000,0.631391,0.822034,0.822267,0.812500,0.832215,0.822086
400,0.201900,0.715514,0.822034,0.821800,0.807947,0.850000,0.807453
450,0.229200,0.746119,0.792373,0.792695,0.773006,0.807692,0.797386



✓ TRAINING COMPLETE!

Final training loss: 0.5410
Total training time: 773.1s


## Evaluate on Validation Set

In [ ]:
print("\n" + "="*70)
print("VALIDATION SET EVALUATION")
print("="*70)

val_results = trainer.evaluate(eval_dataset=val_dataset)

print(f"\n✓ Validation Results:")
print(f"  Accuracy:     {val_results['eval_accuracy']:.3f}")
print(f"  F1 Macro:     {val_results['eval_f1_macro']:.3f}")
print(f"  F1 Positive:  {val_results['eval_f1_positive']:.3f}")
print(f"  F1 Negative:  {val_results['eval_f1_negative']:.3f}")
print(f"  F1 Neutral:   {val_results['eval_f1_neutral']:.3f}")


VALIDATION SET EVALUATION



✓ Validation Results:
  Accuracy:     0.835
  F1 Macro:     0.836
  F1 Positive:  0.848
  F1 Negative:  0.850
  F1 Neutral:   0.810


## Save Fine-tuned Model

In [ ]:
output_dir = '/roberta-large-dialogre-1turn'
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✓ Model saved to: {output_dir}")


✓ Model saved to: ./roberta-large-dialogre-1turn


## Test on 150 Manual Annotations (Gold Standard)

In [ ]:
print("\n" + "="*70)
print("FINAL TEST ON 150 MANUAL ANNOTATIONS")
print("="*70)

test_manual_df = pd.read_csv('/path/to/annotation_samples_150.csv')

test_manual_df['sentence_clean'] = test_manual_df['sentence'].apply(clean_sentence)

# Map labels
test_manual_df['label'] = test_manual_df['manual_label'].map(label_map)

# Convert to dataset
test_dataset = Dataset.from_pandas(test_manual_df[['sentence_clean', 'label']])
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Evaluate
test_results = trainer.evaluate(eval_dataset=test_dataset)

print(f"\n✓ Test Results (150 Manual Annotations):")
print(f"  Accuracy:     {test_results['eval_accuracy']:.3f}")
print(f"  F1 Macro:     {test_results['eval_f1_macro']:.3f}")
print(f"  F1 Positive:  {test_results['eval_f1_positive']:.3f}")
print(f"  F1 Negative:  {test_results['eval_f1_negative']:.3f}")
print(f"  F1 Neutral:   {test_results['eval_f1_neutral']:.3f}")

# Get detailed predictions
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_labels = test_predictions.label_ids

# Classification report
print("\n✓ Detailed Classification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=['positive', 'negative', 'neutral'],
    digits=3
))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_labels, test_preds)

print("\n✓ Confusion Matrix:")
print("                 Predicted")
print("               Pos    Neg    Neu")
print(f"Actual Pos   {cm[0][0]:5d}  {cm[0][1]:5d}  {cm[0][2]:5d}")
print(f"       Neg   {cm[1][0]:5d}  {cm[1][1]:5d}  {cm[1][2]:5d}")
print(f"       Neu   {cm[2][0]:5d}  {cm[2][1]:5d}  {cm[2][2]:5d}")


FINAL TEST ON 150 MANUAL ANNOTATIONS


Map:   0%|          | 0/150 [00:00<?, ? examples/s]


✓ Test Results (150 Manual Annotations):
  Accuracy:     0.633
  F1 Macro:     0.617
  F1 Positive:  0.676
  F1 Negative:  0.638
  F1 Neutral:   0.537

✓ Detailed Classification Report:
              precision    recall  f1-score   support

    positive      0.681     0.671     0.676        70
    negative      0.600     0.682     0.638        44
     neutral      0.581     0.500     0.537        36

    accuracy                          0.633       150
   macro avg      0.621     0.618     0.617       150
weighted avg      0.633     0.633     0.632       150


✓ Confusion Matrix:
                 Predicted
               Pos    Neg    Neu
Actual Pos      47     14      9
       Neg      10     30      4
       Neu      12      6     18


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.utils import resample

def run_bootstrapping_analysis(y_true, y_pred, model_name="Model", n_iterations=1000):
    """
    Calculates 95% Confidence Intervals for Macro F1.
    """
    bootstrapped_f1s = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print(f"Bootstrapping {model_name}...")
    for i in range(n_iterations):
        indices = resample(np.arange(len(y_true)), replace=True)
        resampled_true = y_true[indices]
        resampled_pred = y_pred[indices]

        # Calculate Macro F1 for this sample
        score = f1_score(resampled_true, resampled_pred, average='macro')
        bootstrapped_f1s.append(score)

    # Calculate Confidence Intervals (2.5th and 97.5th percentiles)
    lower = np.percentile(bootstrapped_f1s, 2.5)
    upper = np.percentile(bootstrapped_f1s, 97.5)
    median = np.percentile(bootstrapped_f1s, 50)

    print(f"{model_name} Results:")
    print(f"  Median F1: {median:.3f}")
    print(f"  95% CI:    [{lower:.3f}, {upper:.3f}]")

    return bootstrapped_f1s, (lower, upper)

results_dist, ci_bounds = run_bootstrapping_analysis(test_labels, test_preds, "RoBERTa-1T")

Bootstrapping RoBERTa-1T...
RoBERTa-1T Results:
  Median F1: 0.613
  95% CI:    [0.536, 0.693]


### Save Results

In [ ]:
test_manual_df['roberta_pred'] = [['positive', 'negative', 'neutral'][p] for p in test_preds]
test_manual_df['correct'] = test_manual_df['label'] == test_preds

output_folder = '/path/to/output'

test_manual_df.to_csv(output_folder + 'test_predictions_1turn.csv', index=False)

print("\n✓ Saved: test_predictions_150_manual.csv")

# Save metrics
import json
metrics = {
    'validation': {
        'accuracy': val_results['eval_accuracy'],
        'f1_macro': val_results['eval_f1_macro'],
        'f1_positive': val_results['eval_f1_positive'],
        'f1_negative': val_results['eval_f1_negative'],
        'f1_neutral': val_results['eval_f1_neutral']
    },
    'test_150_manual': {
        'accuracy': test_results['eval_accuracy'],
        'f1_macro': test_results['eval_f1_macro'],
        'f1_positive': test_results['eval_f1_positive'],
        'f1_negative': test_results['eval_f1_negative'],
        'f1_neutral': test_results['eval_f1_neutral']
    },
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_manual_df)
}

with open(output_folder + '1turn_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✓ Saved: finetuned_metrics.json")


✓ Saved: test_predictions_150_manual.csv
✓ Saved: finetuned_metrics.json
